# TutorTune: train, evaluate and download

Select a GPU runtime before proceeding. Upload the original `TutorTune_Project_2.zip` when prompted. This notebook executes the source files in that archive. The included data is a tiny synthetic pipeline check, not proof of improved model quality.

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from pathlib import Path
import zipfile
archives = [Path(name) for name in uploaded if name.endswith(".zip")]
if len(archives) != 1:
    raise ValueError("Upload exactly one project ZIP.")
destination = Path("/content/tutortune_workspace")
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for item in archive.infolist():
        target = (destination / item.filename).resolve()
        if not target.is_relative_to(destination.resolve()):
            raise ValueError("Unsafe archive path")
    archive.extractall(destination)
project = destination / "tutortune"
assert (project / "train.py").exists(), "Expected tutortune/train.py"
%cd $project


## Install training dependencies
If installation requires a runtime restart, restart it, rerun the upload/folder cells as necessary, and continue. Keep a compatible CUDA-enabled Torch build. A free GPU is not guaranteed.

In [ ]:
%pip install -r requirements-train.txt


In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime first.")
print("GPU:", torch.cuda.get_device_name(0))


## Inspect and validate the included data
The archive already contains 24/6/6 sample rows. `make_data.py` can regenerate them, but overwrites those files. Inspect and review any custom data before training.

In [ ]:
!python validate_data.py
from pathlib import Path
print(Path("data/train.jsonl").read_text().splitlines()[0])


## Train
This downloads the small public base model and saves LoRA parameters to `adapter/`. It requires internet and GPU memory. The run is not instant; inspect loss logs and failures rather than assuming success.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "train.py"], check=True)


## Compare base and adapter
This check measures heading format only. Manually inspect correctness, examples and teaching usefulness. A fine-tuned adapter can be worse than its base.

In [ ]:
subprocess.run([sys.executable, "evaluate.py"], check=True)
import json
results = json.loads(Path("evaluation_results.json").read_text())
for row in results[:2]:
    print(row["mode"], row["question"])
    print(row["answer"], "\n")


## Download outputs before ending the session
The archive preserves the entire adapter folder, evaluation results and a run manifest. Extract it into your local `tutortune` project folder, then follow README.md for the comparison app and deployment.

In [ ]:
import zipfile, platform
manifest = {"python": platform.python_version(), "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0)}
Path("run_manifest.json").write_text(json.dumps(manifest, indent=2))
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
Path("training_environment.txt").write_text(freeze)
output = Path("/content/TutorTune_Trained_Artifacts.zip")
with zipfile.ZipFile(output, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in Path("adapter").rglob("*"):
        if path.is_file():
            archive.write(path, path.as_posix())
    for name in ["evaluation_results.json", "run_manifest.json", "training_environment.txt"]:
        archive.write(name, name)
files.download(str(output))
